In [90]:
from __future__ import annotations
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
import os
from dotenv import load_dotenv
import sys
import json
import copy

# Schemas

In [91]:
class GradedObjectiveAnswer(BaseModel):
    question_id: str = Field(description="Unique for objective question (same as in Question Paper)")
    marks_obtained: float = Field(description="Marks obtained => Range(0, total_marks)")
    reasoning: str = Field(description="""
        One (or few) word explanation of why you gave full or less marks like:
        Correct, Incorrect, etc
    """)

In [92]:
class GradedSubjectiveAnswer(BaseModel):
    question_id: str = Field(description="Unique for objective question (same as in Answer Paper)")
    marks_obtained: float = Field(description="""
        Marks obtained => Range(0, total_marks)
        Super question marks is sum of sub question marks (if there are any subquestions)
    """)
    reasoning: str = Field(description="""
        One (or few) word explanation of why you gave full or less marks like:
        Correct, Incorrect, Insufficient, Irrelevant, Too lengthy, Missing, etc.
    """)
    subquestions: Optional[List[GradedSubjectiveAnswer]] = Field(
        default=None, 
        description="""
        Nested list of subquestion evaluations for child parts (Q(n(m)), Q(n(m(o))), ...).
        This field MUST either contain child GradedSubjectiveAnswer objects OR be null/None. 
        """
    )

GradedSubjectiveAnswer.model_rebuild()

In [93]:
class GradedPaper(BaseModel):
    roll_no: str = Field(description="Roll number of student")
    name: str | None = Field(default=None, description="Name of student (if available)")
    objective: List[GradedObjectiveAnswer] | None = Field(default=None, description="Collectection of graded MCQs")
    subjective : List[GradedSubjectiveAnswer] | None = Field(default=None, description="Collection of graded answers of questions and subquestions")

GradedPaper.model_rebuild()

# Grader

In [94]:
class ExamGrader:
    def __init__(self, gemini_client: genai.Client):
            self.client = gemini_client
    
    def grade_exam_paper(self, exam_paper_json: str, answer_paper_json: str, instructions: str = "",  model: str = "gemini-3.5-flash-lite"):
        prompt = f"""
            You have to grade an exam answer paper and return response in desired format.
            Exam paper: {exam_paper_json}
            Answer paper: {answer_paper_json}
            Instructions : {instructions}
            CRITICAL STRUCTURAL & HIERARCHY RULES:
            1. STRICT MATCHING: Map answers directly to the corresponding question IDs from the Exam Paper.
            2. PRESERVE HIERARCHY: If in Exam Paper/ Anwer Paper, there is nesting of subquestions, same should be reflected in Graded Exam Paper.
            3. EMPTY VS NULL RULE: If a question has subquestions, populate the `subquestions` array with child `GradedSubjectiveAnswer` objects. 
            4. MARKS AGGREGATION: The `marks_obtained` for a parent question MUST equal the sum of `marks_obtained` across all its inner `subquestions`.
            5. INSTRUCTIONS: Follow (if any) strictly. Don't be too generous or strict unless explicitly asked.
        """
        response = self.client.interactions.create(
             model=model,
             input=prompt,
             response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": self.inline_schema(GradedPaper)
             },
        )
        formatted_graded_exam = GradedPaper.model_validate_json(response.output_text)
        return formatted_graded_exam.model_dump_json()

    def inline_schema(self, model, property_name="subquestions", depth=4):
        schema = model.model_json_schema()
        defs = schema.pop("$defs", {})

        def resolve(node, d):
            if isinstance(node, dict):
                if "$ref" in node:
                    target = copy.deepcopy(defs[node["$ref"].split("/")[-1]])
                    if d <= 1:  # last level: no further nesting
                        target.get("properties", {}).pop(property_name, None)
                    return resolve(target, d - 1)
                return {k: resolve(v, d) for k, v in node.items()}
            if isinstance(node, list):
                return [resolve(i, d) for i in node]
            return node

        return resolve(schema, depth)
            

## Testing

In [95]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [96]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

In [97]:
exam_grader = ExamGrader(client)

In [98]:
exam_paper_json = '{"subject":"DATA STRUCTURES & ALGORITHMS","course_id":null,"date":null,"total_marks":50.0,"objective":[{"question_id":"Q1","marks":2.0,"mcq_statement":"What is the worst-case time complexity of binary search on a sorted array of n elements?\\n(A) O(n) (B) O(log n) (C) O(n log n) (D) O(1)"},{"question_id":"Q2","marks":2.0,"mcq_statement":"Which data structure uses LIFO (Last In First Out) order?\\n(A) Queue (B) Stack (C) Linked List (D) Tree"},{"question_id":"Q3","marks":2.0,"mcq_statement":"In a max-heap containing n elements, where is the largest element always located?\\n(A) At the last leaf node (B) At the root (C) At any leaf node (D) At the middle index"},{"question_id":"Q4","marks":2.0,"mcq_statement":"What is the time complexity of inserting an element at the beginning of a singly linked list?\\n(A) O(1) (B) O(n) (C) O(log n) (D) O(n^2)"},{"question_id":"Q5","marks":2.0,"mcq_statement":"Which traversal of a Binary Search Tree visits nodes in ascending sorted order?\\n(A) Pre-order (B) Post-order (C) In-order (D) Level-order"},{"question_id":"Q6","marks":2.0,"mcq_statement":"Which sorting algorithm has the best average-case time complexity?\\n(A) Bubble Sort (B) Insertion Sort (C) Merge Sort (D) Selection Sort"},{"question_id":"Q7","marks":2.0,"mcq_statement":"A graph with no cycles and n-1 edges connecting n vertices is called a:\\n(A) Complete graph (B) Tree (C) Bipartite graph (D) Directed acyclic multigraph"},{"question_id":"Q8","marks":2.0,"mcq_statement":"Which data structure is most suitable for implementing a priority queue efficiently?\\n(A) Array (B) Linked List (C) Heap (D) Stack"},{"question_id":"Q9","marks":2.0,"mcq_statement":"What is the space complexity of a recursive Fibonacci function without memoization (in terms of call stack depth for input n)?\\n(A) O(1) (B) O(n) (C) O(log n) (D) O(2^n)"},{"question_id":"Q10","marks":2.0,"mcq_statement":"Which technique does Dijkstra\'s algorithm use to find the shortest path?\\n(A) Dynamic Programming only (B) Greedy approach (C) Backtracking (D) Divide and Conquer"}],"subjective":[{"question_id":"Q11","marks":4.0,"question_statement":"Differentiate between an array and a linked list. Give one real-world scenario where a linked list would be preferred over an array.","subquestions":null},{"question_id":"Q12","marks":4.0,"question_statement":"Explain what a hash collision is and briefly describe one method used to resolve it.","subquestions":null},{"question_id":"Q13","marks":4.0,"question_statement":"Write the recurrence relation for the Merge Sort algorithm and state its time complexity using this relation.","subquestions":null},{"question_id":"Q14","marks":4.0,"question_statement":"What is the difference between BFS (Breadth-First Search) and DFS (Depth-First Search)? Mention one use case for each.","subquestions":null},{"question_id":"Q15","marks":4.0,"question_statement":"Define \\"dynamic programming\\" and explain how it differs from plain recursion.","subquestions":null},{"question_id":"Q16","marks":10.0,"question_statement":"Consider an unsorted array of integers: [38, 27, 43, 3, 9, 82, 10].","subquestions":[{"question_id":"Q16(a)","marks":6.0,"question_statement":"Trace through the Merge Sort algorithm step-by-step to sort this array, showing the divide and merge steps.","subquestions":null},{"question_id":"Q16(b)","marks":4.0,"question_statement":"State the time complexity of Merge Sort in the best, average, and worst case, and briefly justify why all three cases are the same.","subquestions":null}]}]}'
answer_paper_json = '{"subject":"DATA STRUCTURES & ALGORITHMS","roll_no":"DSA-033","name":"Sara Malik","objective":[{"question_id":"Q1","chosen_option":"B"},{"question_id":"Q2","chosen_option":"C"},{"question_id":"Q3","chosen_option":"B"},{"question_id":"Q4","chosen_option":"B"},{"question_id":"Q5","chosen_option":"C"},{"question_id":"Q6","chosen_option":"B"},{"question_id":"Q7","chosen_option":"B"},{"question_id":"Q8","chosen_option":"C"},{"question_id":"Q9","chosen_option":"D"},{"question_id":"Q10","chosen_option":"B"}],"subjective":[{"question_id":"Q11","answer_statement":"An array is a fixed-size contiguous block of memory allowing O(1) random access but O(n) insertion/deletion in the middle. A linked list is a dynamic collection of nodes connected via pointers, giving O(1) insertion/deletion at the head but O(n) access time. Linked lists are preferred for implementing something like a browser\'s history (back/forward navigation) where frequent insertion and deletion happens at the ends.","subanswers":null},{"question_id":"Q12","answer_statement":null,"subanswers":null},{"question_id":"Q13","answer_statement":"T(n) = 2T(n/2) + O(n), which by the master theorem gives O(n log n).","subanswers":null},{"question_id":"Q14","answer_statement":"BFS uses a queue and explores all neighbors at the current depth before going deeper, so it is used to find the shortest path in an unweighted graph. DFS uses a stack or recursion and explores as far down one path as possible before backtracking, so it is used for topological sorting and detecting cycles in a graph.","subanswers":null},{"question_id":"Q15","answer_statement":null,"subanswers":null},{"question_id":"Q16","answer_statement":null,"subanswers":[{"question_id":"Q16(a)","answer_statement":"[38, 27, 43, 3, 9, 82, 10] is split into [38, 27, 43] and [3, 9, 82, 10]. The first splits into [38] and [27, 43], which further splits into [27] and [43]. Merging [27] and [43] gives [27, 43], then merging with [38] gives [27, 38, 43]. The second half [3, 9, 82, 10] splits into [3, 9] and [82, 10]. [3, 9] merges to [3, 9] and [82, 10] merges to [10, 82]. Merging [3, 9] and [10, 82] gives [3, 9, 10, 82]. Finally merging [27, 38, 43] and [3, 9, 10, 82] gives [3, 9, 10, 27, 38, 43, 82].","subanswers":null},{"question_id":"Q16(b)","answer_statement":"Best, average and worst case are all O(n log n) because merge sort always divides the array exactly in half regardless of the values or their order, and the merge step always takes linear time O(n), so the total work per level is O(n) across log n levels no matter how the input is arranged.","subanswers":null}]}]}'

In [99]:
instructions = """
GRADING INSTRUCTIONS FOR THE AI:
1. MCQs: binary, full 2 marks or 0, no partial credit.
2. Short answers: grade against rubric sub-points; accept different wording if the concept is correct.
3. Long answer: grade (a) and (b) separately; give partial credit for a correct final array even with an incomplete trace.
4. Do not penalize spelling/grammar - only factual/conceptual errors.
5. Output per-question score plus total out of 50."""

In [100]:
graded_exam_json = exam_grader.grade_exam_paper(
    exam_paper_json,
    answer_paper_json, 
    instructions
)

In [101]:
json.loads(graded_exam_json)

{'roll_no': 'DSA-033',
 'name': 'Sara Malik',
 'objective': [{'question_id': 'Q1',
   'marks_obtained': 2.0,
   'reasoning': 'Correct'},
  {'question_id': 'Q2', 'marks_obtained': 0.0, 'reasoning': 'Incorrect'},
  {'question_id': 'Q3', 'marks_obtained': 2.0, 'reasoning': 'Correct'},
  {'question_id': 'Q4', 'marks_obtained': 0.0, 'reasoning': 'Incorrect'},
  {'question_id': 'Q5', 'marks_obtained': 2.0, 'reasoning': 'Correct'},
  {'question_id': 'Q6', 'marks_obtained': 0.0, 'reasoning': 'Incorrect'},
  {'question_id': 'Q7', 'marks_obtained': 2.0, 'reasoning': 'Correct'},
  {'question_id': 'Q8', 'marks_obtained': 0.0, 'reasoning': 'Incorrect'},
  {'question_id': 'Q9', 'marks_obtained': 0.0, 'reasoning': 'Incorrect'},
  {'question_id': 'Q10', 'marks_obtained': 2.0, 'reasoning': 'Correct'}],
 'subjective': [{'question_id': 'Q11',
   'marks_obtained': 4.0,
   'reasoning': 'Correct',
   'subquestions': None},
  {'question_id': 'Q12',
   'marks_obtained': 0.0,
   'reasoning': 'Missing',
   'sub